In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Sanjay_Chandra_vs_Cbi_on_23_November_2011.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Sakiri_Vasu_vs_State_Of_U_P_And_Others_on_7_December_2007.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/National_Insurance_Co_Ltd_vs_Pranay_Sethi_on_31_October_2017.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Arnesh_Kumar_vs_State_Of_Bihar_Anr_on_2_July_2014.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Sarla_Verma_Ors_vs_Delhi_Transport_Corp_Anr_on_15_April_2009.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Lata_Singh_vs_State_Of_U_P_Another_on_7_July_2006.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Lalita_Kumari_vs_Govt_Of_U_P_Ors_on_12_November_2013.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Pareshkumar_Jaykarbhai_Brahmbhatt_vs_State_Of_Gujarat

In [9]:
!pip install datasets transformers rouge-score bert-score -q

In [10]:
import requests
from bs4 import BeautifulSoup
import os
import time

os.makedirs('/kaggle/working/judgements', exist_ok=True)

# Indian Kanoon API - free for research
# Search for Supreme Court cases
url = "https://indiankanoon.org/search/?formInput=supreme+court+contract&pagenum=1"

headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# Get all case links
links = soup.find_all('a', href=True)
case_links = [l['href'] for l in links if '/doc/' in l['href']]
print(f"Found {len(case_links)} cases")
print(case_links[:10])

Found 0 cases
[]


In [11]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Sanjay_Chandra_vs_Cbi_on_23_November_2011.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Sakiri_Vasu_vs_State_Of_U_P_And_Others_on_7_December_2007.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/National_Insurance_Co_Ltd_vs_Pranay_Sethi_on_31_October_2017.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Arnesh_Kumar_vs_State_Of_Bihar_Anr_on_2_July_2014.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Sarla_Verma_Ors_vs_Delhi_Transport_Corp_Anr_on_15_April_2009.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Lata_Singh_vs_State_Of_U_P_Another_on_7_July_2006.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Lalita_Kumari_vs_Govt_Of_U_P_Ors_on_12_November_2013.PDF
/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Pareshkumar_Jaykarbhai_Brahmbhatt_vs_State_Of_Gujarat

In [12]:
!pip install transformers rouge-score bert-score PyPDF2 -q

In [13]:
from transformers import pipeline
import torch

print("GPU available:", torch.cuda.is_available())

print("Loading DistilBART...")
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=0 if torch.cuda.is_available() else -1
)
print("Model loaded successfully!")

GPU available: True
Loading DistilBART...


KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration
import torch

print("GPU available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading DistilBART...")
model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
print("Model loaded successfully!")

In [ ]:
import PyPDF2
import os

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + " "
    return ' '.join(text.split())

def summarize_document(text):
    if len(text) < 100:
        return "Document too short"
    
    # Truncate to first 3000 characters
    if len(text) > 3000:
        text = text[:3000]
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(device)
    
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=150,
        min_length=50,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )
    
    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )
    return summary

# Your exact path
pdf_folder = '/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/'

pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.PDF') or f.endswith('.pdf')]
print(f"Found {len(pdf_files)} PDFs\n")

generated_summaries = []
documents = []
filenames = []

for i, filename in enumerate(pdf_files):
    pdf_path = os.path.join(pdf_folder, filename)
    print(f"Processing {i+1}/{len(pdf_files)}: {filename}")
    
    text = extract_text_from_pdf(pdf_path)
    print(f"Extracted {len(text)} characters")
    
    if len(text) < 100:
        print("Skipping - too short\n")
        continue
    
    summary = summarize_document(text)
    generated_summaries.append(summary)
    documents.append(text)
    filenames.append(filename)
    
    print(f"Summary: {summary[:150]}...\n")

print(f"Successfully processed {len(generated_summaries)}/{len(pdf_files)} documents")

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for doc, gen in zip(documents, generated_summaries):
    reference = ' '.join(doc.split()[:150])
    scores = scorer.score(reference, gen)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

print("=" * 50)
print("ROUGE SCORES")
print("=" * 50)
print(f"Documents evaluated : {len(rouge1_scores)}")
print(f"ROUGE-1             : {sum(rouge1_scores)/len(rouge1_scores):.4f}")
print(f"ROUGE-2             : {sum(rouge2_scores)/len(rouge2_scores):.4f}")
print(f"ROUGE-L             : {sum(rougeL_scores)/len(rougeL_scores):.4f}")

In [ ]:
from bert_score import score as bert_score

references = [' '.join(doc.split()[:150]) for doc in documents]

print("Calculating BERTScore...")
P, R, F1 = bert_score(
    generated_summaries,
    references,
    lang="en",
    verbose=True
)

print("\n" + "=" * 50)
print("BERTSCORE")
print("=" * 50)
print(f"Precision : {P.mean():.4f}")
print(f"Recall    : {R.mean():.4f}")
print(f"F1        : {F1.mean():.4f}")

In [ ]:
print("\n" + "=" * 60)
print("SAMPLE SUMMARIES")
print("=" * 60)

for i in range(len(filenames)):
    case_name = filenames[i].replace('_', ' ').replace('.PDF', '')
    print(f"\nCase {i+1}: {case_name}")
    print(f"Summary: {generated_summaries[i]}")
    print("-" * 60)

In [14]:
import PyPDF2

# Extract full text
pdf_path = '/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Pareshkumar_Jaykarbhai_Brahmbhatt_vs_State_Of_Gujarat_on_15_December_2017.PDF'

with open(pdf_path, 'rb') as f:
    reader = PyPDF2.PdfReader(f)
    full_text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            full_text += extracted + " "

full_text = ' '.join(full_text.split())
print(f"Total characters: {len(full_text)}")
print(f"Total words: {len(full_text.split())}")

# Skip header - start from 20% into document
start = len(full_text) // 5
chunk = full_text[start:start + 3000]

print("\nChunk being sent to model:")
print(chunk[:500])

Total characters: 129506
Total words: 21941

Chunk being sent to model:
 would suggest that the powers of the Court for release of the vehicle at an interim stage are curtailed. 16 Having heard the learned counsel appearing for the parties and having considered the materials on record, the only question that falls for my consideration is whether Section 98(2) of the Gujarat Prohibition Act restricts or bars the jurisdiction of a Magistrate or Court to pass an order under Section 451 of the Cr.P.C. to release the vehicle by way of interim custody pending the investig


In [15]:
# Summarize just this chunk
chunk = full_text[start:start + 3000]

inputs = tokenizer(
    chunk,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=200,
    min_length=80,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("GENERATED SUMMARY:")
print(summary)

NameError: name 'tokenizer' is not defined

In [16]:
from transformers import BartTokenizer, BartForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_name = "sshleifer/distilbart-cnn-12-6"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
print("Model loaded!")

Device: cuda


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded!


In [17]:
import PyPDF2

pdf_path = '/kaggle/input/datasets/suhanichhabra123/indian-legal-documents-summ/Pareshkumar_Jaykarbhai_Brahmbhatt_vs_State_Of_Gujarat_on_15_December_2017.PDF'

with open(pdf_path, 'rb') as f:
    reader = PyPDF2.PdfReader(f)
    full_text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            full_text += extracted + " "

full_text = ' '.join(full_text.split())

# Skip header - start from 20% into document
start = len(full_text) // 5
chunk = full_text[start:start + 3000]

# Summarize
inputs = tokenizer(
    chunk,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=200,
    min_length=80,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("GENERATED SUMMARY:")
print(summary)

GENERATED SUMMARY:
 The anti-social elements adulterate liquor by mixing the methyl alcohol or other poisonous substances and make the spurious liquor which is commonly known as Laththa . Recently, due to such type of illegal activities, some people have lost their lives . With a view to prohibiting the misuse of such illicit and spurious liquor, it is considered necessary to amend the existing provisions of the Bombay Prohibition Act, 1949 and make stringent provisions for offences relating to manufacturing, constructing, selling, buying, keeping, keeping and transporting, etc. of such spurious liquor .


In [18]:
# Try starting at 40% into document
start = int(len(full_text) * 0.4)
chunk = full_text[start:start + 3000]

print("Chunk preview:")
print(chunk[:300])
print("\n---\n")

inputs = tokenizer(
    chunk,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=200,
    min_length=80,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("GENERATED SUMMARY:")
print(summary)

Chunk preview:
iew has been concluded by the Supreme Court in Bhim Sen vs. State of U.P. [AIR 1955 SC 435]. 32 Section 5, Cr.P.C. deals with the saving clause and runs as follows: "Nothing contained in this Code shall, in the absence of a specific provision to the contrary, affect any special or local law for the 

---

GENERATED SUMMARY:
 Section 5, Cr.P.C. deals with the saving clause and runs as follows: "Nothing contained in this Code shall, in the absence of a specific provision to the contrary, affect any special or local law for the time being in force, or any special jurisdiction or power conferred" The anatomy of this section is simple, yet subtle . It seems clear that this specific provision must not be in the Criminal Procedure Code itself .


In [19]:
# Try last 30% - where the final judgment usually is
start = int(len(full_text) * 0.70)
chunk = full_text[start:start + 3000]

print("Chunk preview:")
print(chunk[:300])
print("\n---\n")

inputs = tokenizer(
    chunk,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=200,
    min_length=80,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("GENERATED SUMMARY:")
print(summary)

Chunk preview:
r. But some tests which the Courts have evolved have worked effectively and with reasonable assurance of success. 55 Furthermore in the Presidential Election Case reported in AIR 1974 SC 1682, the Hon'ble Chief Justice of the Apex Court speaking on behalf of a seven−Judge Bench had specifically held

---

GENERATED SUMMARY:
 In the Presidential Election Case reported in AIR 1974 SC 1682, the Hon'ble Chief Justice of the Apex Court speaking on behalf of a seven-Judge Bench had specifically held as follows (Para 13) :Pareshkumar Jaykarbhai Brahmbhatt vs State Of Gujarat on 15 December, 2017 Indian Kanoon - http://indiankanoon.org/doc/50572539/


In [20]:
# Try last 15% - final judgment and conclusion
start = int(len(full_text) * 0.85)
chunk = full_text[start:start + 3000]

print("Chunk preview:")
print(chunk[:300])
print("\n---\n")

inputs = tokenizer(
    chunk,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to(device)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=200,
    min_length=80,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("GENERATED SUMMARY:")
print(summary)

Chunk preview:
ds as under: "482. Saving of inherent powers of High Court − Nothing in this Code shall be deemed to limit or affect the inherent powers of the High Court to make such orders as may be necessary to give effect to any order under this Code, or to prevent abuse of the process of any Court or otherwise

---

GENERATED SUMMARY:
 The inherent powers of the High Court is saved only in a case where an order has been passed by the Criminal Court which is required to be set aside to secure the ends of justice . The Supreme Court in the case of West Bengal and others vs. Sujit Kumar Rana [(2004) 4 SCC 129] has taken this view as contained in paras 32 to 46. The High Court cannot, thus, in such a situation exercise its jurisdiction under Section 482 of the Code of Criminal Procedure . Once it is held that the criminal Court had no power to deal with the property seized under the Act, the question of High Court exercising its jurisdiction would not arise .
